# Random Forests & Extra Trees: Ensembling, Feature Importance, and Permutation Testing
Accompanies **Unit 5.2: Bagging, Random Forests and Extra Trees**.

The variance of an ensemble of $B$ trees with individual variance $\sigma^2$ and pairwise correlation $\rho$ is:
$$\text{Var}(\bar{f}) = \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2$$

Random Forests drive ensemble variance down through two orthogonal mechanisms:
1. **Bootstrap Aggregation (Bagging)**: Each tree trains on an independent bootstrap sample with replacement.
2. **Feature Subsampling ($m = \sqrt{p}$)**: At each split, only a random subset of features is evaluated, decorrelating the trees (reducing $\rho$).

In this notebook, we tune Random Forest hyperparameters and compare **Mean Decrease in Impurity (MDI)** against **Permutation Feature Importance** on holdout test data.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.inspection import permutation_importance
from scipy.stats import randint

data_path = os.path.abspath('../../data/diabetic_data.csv')
df = pd.read_csv(data_path)

numeric_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 
                'num_medications', 'number_outpatient', 'number_emergency', 
                'number_inpatient', 'number_diagnoses']
X = df[numeric_cols].fillna(0)
y = (df['readmitted'] == '<30').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Hyperparameter Tuning Rationale:
- `n_estimators`: Adding more trees *never* causes overfitting in Random Forests. It monotonically reduces variance until reaching an asymptotic plateau (typically $100-300$ trees).
- `max_features`: Evaluating fewer features per split decorrelates trees (lowers $\rho$), but setting it too small degrades individual tree strength. Standard default is $\sqrt{p}$.
- `min_samples_leaf`: Enforces smoothing on leaf predictions.

In [ ]:
param_dist = {
    'n_estimators': randint(80, 220),
    'max_depth': randint(5, 16),
    'max_features': ['sqrt', 'log2', 0.5],
    'min_samples_leaf': randint(2, 10)
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, oob_score=True, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=15,
    cv=3,
    random_state=42,
    n_jobs=-1
)
rf_search.fit(X_train, y_train)

print(f"Best CV ROC-AUC: {rf_search.best_score_:.4f}")
print(f"Optimal Parameters: {rf_search.best_params_}")
best_rf = rf_search.best_estimator_
print(f"Out-Of-Bag (OOB) Accuracy: {best_rf.oob_score_:.4f}")

### Comparing Feature Importances: Impurity (MDI) vs. Permutation
- **MDI (Impurity)**: Measures total Gini reduction in training trees. Vulnerable to continuous feature bias.
- **Permutation Importance**: Shuffles column $j$ on **unseen test data** and measures the drop in accuracy. Model-agnostic and immune to in-sample overfitting!

In [ ]:
# 1. MDI Feature Importance
mdi_importances = best_rf.feature_importances_

# 2. Permutation Importance on holdout test set
perm_result = permutation_importance(best_rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
perm_importances = perm_result.importances_mean

# Comparison plot
plt.figure(figsize=(11, 4.5))
x_indices = np.arange(len(numeric_cols))
bar_width = 0.35

plt.bar(x_indices - bar_width/2, mdi_importances, width=bar_width, label='MDI (In-Sample Impurity)', color='royalblue')
plt.bar(x_indices + bar_width/2, perm_importances, width=bar_width, label='Permutation (Test Holdout Drop)', color='crimson')

plt.xticks(x_indices, numeric_cols, rotation=35, ha='right', fontsize=9)
plt.ylabel('Importance Score', fontsize=11)
plt.title('MDI vs. Permutation Feature Importance Comparison', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()